In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

project_path = '/content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1'
os.chdir(project_path)

print("Current Directory:", os.getcwd())

Current Directory: /content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1


In [3]:
import os

# Set your Informer data directory
informer_path = f'{project_path}/Informer2020-original'
data_dir = f'{informer_path}/data/ETT'
os.makedirs(data_dir, exist_ok=True)

# File path
data_file = f'{data_dir}/ETTh1.csv'

# Download ETTh1 dataset
if not os.path.exists(data_file):
    print("Downloading ETTh1.csv...")
    os.system(
        f'wget -q "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv" '
        f'-O "{data_file}"'
    )

# Verify
if os.path.exists(data_file):
    print(f"✅ Download successful: {data_file}")
else:
    print("❌ Download failed.")

✅ Download successful: /content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1/Informer2020-original/data/ETT/ETTh1.csv


In [4]:
# Fix np.Inf -> np.inf in tools.py (removed in NumPy 2.0)
tools_path = f'{project_path}/Informer2020-original/utils/tools.py'

with open(tools_path, 'r') as f:
    content = f.read()

content_fixed = content.replace('np.Inf', 'np.inf')

with open(tools_path, 'w') as f:
    f.write(content_fixed)

print("✅ Fixed np.Inf -> np.inf in utils/tools.py")

✅ Fixed np.Inf -> np.inf in utils/tools.py


In [5]:
!bash /content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1/experiments/exp6_lod_post/exp6_lod_post_phase1.sh

 Experiment 6 LOD Post-Softmax — Phase 1: Alpha Screening
 Date: Sun Aug 16 09:15:39 AM UTC 2026
------------------------------------------------------------
 Architecture:
   Q, K  ← combined_emb  (value + temporal + legendre)
   V     ← delta_x       (x_i - x_{i-1}, no dropout)
   weights = softmax(Q·K^T / sqrt(d))   [softmax first]
   alpha   = 1/(1 + |i-j|^decay_a)       [post-softmax]
   output  = (weights * alpha) · V
------------------------------------------------------------
 Alpha sweep:   decay_a in {0.5, 1.0, 2.0}
 Pred lens:     96, 192
 Seed:          2021 (Phase 1 only)
 Total runs:    6  (3 alpha × 2 pred_len × 1 seed)
------------------------------------------------------------
 Bug fixes applied before run:
   FINDING F: InformerStack decay_a threading + forward() tuple fix
   FINDING G: delta_x downsampling through ConvLayer
   FINDING H: exp_informer.py now forwards --decay_a to model
------------------------------------------------------------
 Reference results (E

Alpha 0.5 had best performence and hence was carried forward into phase 2

In [6]:
!bash /content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1/experiments/exp6_lod_post/exp6_lod_post_ph2.sh

 Experiment 6 LOD Post-Softmax — Phase 2: Stability Validation (decay_a=0.5)
 Date: Sun Aug 16 09:54:19 AM UTC 2026
------------------------------------------------------------
 Architecture:
   Q, K  ← combined_emb  (value + temporal + legendre)
   V     ← delta_x       (x_i - x_{i-1}, no dropout)
   weights = softmax(Q·K^T / sqrt(d))   [softmax first]
   alpha   = 1/(1 + |i-j|^decay_a)       [post-softmax]
   output  = (weights * alpha) · V
------------------------------------------------------------
 decay_a fixed at 0.5
 Pred lens:     48, 96, 192, 336
 Seeds:         2021, 2022, 2023
 Total runs:    12  (4 pred_lens × 3 seeds)
------------------------------------------------------------
 Difference vs Exp6-Pre:
   Pre:  scores = scores * alpha → softmax → weights  [stronger suppression]
   Post: softmax → weights → weights * alpha           [softer, linear]

Copying model files to /content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1/Informer2020-original/models/ ...
File copy complet